#***Model Training***

In [ ]:
NUM_CLASSES = len(classes)

# Load MobileNetV2 pre-trained on ImageNet (without top classification layer)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze the base model weights

# Build the full model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),      # Convert feature maps to a single vector
    BatchNormalization(),          # stabilise activations
    Dense(256, activation='relu'), # Fully connected layer
    Dropout(0.5),                  # Dropout to prevent overfitting
    Dense(128, activation='relu'),
    Dropout(0.30),
    Dense(NUM_CLASSES, activation='softmax')  # final output layer
], name="SortWise_MobileNetV2")

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel compiled ✅")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Model compiled ✅


> * ***Setting up callbacks (Hooks) for monitoring & intervention during training***


In [ ]:
Hooks = [
    tf.keras.callbacks.ModelCheckpoint(   # 1st. save the best version based on verification accuracy
        filepath='best_model.weights.h5',
        save_weights_only=True,
        monitor='val_accuracy',
        save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(     # 2nd. Stop training early if the model stops improving
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau( # 3rd. Reduce learning rate when val_loss plateaus
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(       # 4th. Record data for TensorBoard monitoring
        log_dir='./logs'
    )
]

In [ ]:
# Train the model — Phase 1: train only the custom head (base frozen)
print("Training Phase 1: Custom head only (base model frozen)...")
print("=" * 55)

history = model.fit(
    train_generator,
    epochs=15,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=Hooks,
    class_weight=class_weights,  # Handle class imbalance
    verbose=1
)

print("\nDone — Phase 1.")

Training Phase 1: Custom head only (base model frozen)...
Epoch 1/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 63s 560ms/step - accuracy: 0.7669 - loss: 0.6377 - val_accuracy: 0.8734 - val_loss: 0.3718 - learning_rate: 1.0000e-05
Epoch 2/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 75s 497ms/step - accuracy: 0.7785 - loss: 0.5997 - val_accuracy: 0.8813 - val_loss: 0.3593 - learning_rate: 1.0000e-05
Epoch 3/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 56s 501ms/step - accuracy: 0.7842 - loss: 0.5897 - val_accuracy: 0.8786 - val_loss: 0.3478 - learning_rate: 1.0000e-05
Epoch 4/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 53s 481ms/step - accuracy: 0.7909 - loss: 0.5924 - val_accuracy: 0.8760 - val_loss: 0.3428 - learning_rate: 1.0000e-05
Epoch 5/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 53s 481ms/step - accuracy: 0.8164 - loss: 0.4990 - val_accuracy: 0.8799 - val_loss: 0.3348 - learning_rate: 1.0000e-05
Epoch 6/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 56s 501ms/step - accuracy: 0.8071 - loss: 0.5415 - val_accuracy: 0.8905 - val_loss: 0.3253 - learning_rate:

In [ ]:
# ── Fine-tuning: unfreeze the last 30 layers of MobileNetV2 ──────────────────
print("\nFine-tuning Phase 2: Unfreezing last 30 layers of base model...")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Re-compile with a much lower learning rate to avoid destroying learned weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    epochs=10,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=Hooks,
    class_weight=class_weights,
    verbose=1
)

# ── Merge both history objects for unified plots later ────────────────────────
for key in history.history:
    history.history[key] += history_fine.history[key]

print("\nDone — Phase 2 (fine-tuning).")

# Save the trained model
model.save("best_model.h5")
print("\nModel saved as best_model.h5")


Fine-tuning Phase 2: Unfreezing last 30 layers of base model...
Epoch 1/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 90s 638ms/step - accuracy: 0.8574 - loss: 0.3903 - val_accuracy: 0.9156 - val_loss: 0.2609 - learning_rate: 1.0000e-05
Epoch 2/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 55s 496ms/step - accuracy: 0.8645 - loss: 0.3755 - val_accuracy: 0.9156 - val_loss: 0.2556 - learning_rate: 1.0000e-05
Epoch 3/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 55s 494ms/step - accuracy: 0.8710 - loss: 0.3482 - val_accuracy: 0.9195 - val_loss: 0.2490 - learning_rate: 1.0000e-05
Epoch 4/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 54s 490ms/step - accuracy: 0.8795 - loss: 0.3403 - val_accuracy: 0.9248 - val_loss: 0.2385 - learning_rate: 1.0000e-05
Epoch 5/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 54s 483ms/step - accuracy: 0.8798 - loss: 0.3222 - val_accuracy: 0.9235 - val_loss: 0.2321 - learning_rate: 1.0000e-05
Epoch 6/10
111/111 ━━━━━━━━━━━━━━━━━━━━ 55s 497ms/step - accuracy: 0.8874 - loss: 0.3012 - val_accuracy: 0.9274 - val_loss: 0.2264 - learnin


Done — Phase 2 (fine-tuning).

Model saved as best_model.h5


In [ ]:
# Download test folder
shutil.make_archive("test_data", "zip", "split_dataset/test")
files.download("test_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# download the best model
files.download('best_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>